In [ ]:
import numpy as np
from adios2 import FileReader

output = "/home/hadis/custom_vector/buildParticleOriented/buildVS/jan28_dotTestAdios/outputs/run_1.bp"

with FileReader(output) as s:
    # inspect variables
    # vars = s.available_variables()
    # for name, info in vars.items():
    #     print("variable_name: " + name, end=" ")
    #     for key, value in info.items():
    #         print("\t" + key + ": " + value, end=" ")
    #     print()
    # print()

    positions = s.read("positions", step_selection=[0, 10])
        # read variables return a numpy array with corresponding selection
    steps = int(vars['positions']['AvailableStepsCount'])
    print(steps)
    positions = s.read("positions", step_selection=[0, steps])
    velocities = s.read("velocities", step_selection=[0, steps])
    print(f"positions shape is {positions.shape}, velocities shape is {velocities.shape}")
    # positions.appent(np.array([1,3]))
    positions2 = np.append(positions, velocities, axis=0)
    print(positions2.shape)


    # steps = int(vars['temperature']['AvailableStepsCount'])
    # temperature = s.read("temperature", step_selection=[0, steps])
    temperature = s.read_attribute("temperature")
    print(f"temperature array size is {temperature.size} of shape {temperature.shape}")
    print(f"temperature unit is {temperature} of type {type(temperature)}")
    print(f"temperature is {temperature}")

In [ ]:
output = "/home/hadis/custom_vector/buildParticleOriented/buildVS/jan28_dotTestAdios/outputs/run_1.bp"
positions = np.zeros((1000000,2))
for _ in range(200):
    with FileReader(output) as s:
        steps = int(vars['positions']['AvailableStepsCount'])
        positions = s.read("positions", step_selection=[0, steps])
        positions = np.append(positions, positions, axis=0)

In [ ]:
positions.shape

In [ ]:
import adios2
import sys

print("Python version:", sys.version)
print("\nADIOS2 module contents:")
print(dir(adios2))

try:
    print("\nTrying direct file opening:")
    with adios2.File("/home/hadis/custom_vector/buildParticleOriented/buildVS/feb1_testAdios/outputs/run_0.bp", "r") as f:
        print("File opened successfully")
        print("Available methods:", dir(f))
except Exception as e:
    print("Error:", e)

In [ ]:
outputDir = "/home/hadis/custom_vector/buildParticleOriented/buildVS/jan28_dotTestAdios/outputs/"
plot_neighbors(outputDir)

In [5]:
file_path = "/home/hadis/custom_vector/buildParticleOriented/buildVS/feb1_testAdios/outputs/run_0.bp"


In [ ]:
import numpy as np
from adios2 import FileReader

s = FileReader(file_path)
v = s.read("positions", step_selection=[0, 100])
s.available_variables()["positions"]

In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from adios2 import FileReader

output_dir = "/home/hadis/custom_vector/buildParticleOriented/buildVS/jan28_dotTestAdios/outputs/"

pattern = re.compile("run_([0-9]+).bp")
run_numbers = [int(pattern.match(x)[1]) for x in os.listdir(output_dir) if pattern.match(x)]
 
run_numbers.sort()

neighbors_data = []

for run_num in run_numbers:
    file_path = os.path.join(output_dir, f"run_{run_num}.bp")
    with FileReader(file_path) as reader:
        if "number of neighbors" in reader.available_variables():
            var_info = reader.available_variables()["number of neighbors"]
            steps = int(var_info.get("AvailableStepsCount", 1))
            
            data = reader.read("number of neighbors", step_selection=[steps - 5, 1])
            data = data.flatten()
            neighbors_data.append(data)
        else:
            neighbors_data.append(np.full(7, np.nan))

# Return all data as a NumPy array.
np.array(neighbors_data)

In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from adios2 import FileReader
from scipy.signal import savgol_filter

def read_neighbors(output_dir):
    pattern = re.compile("run_([0-9]+).bp")
    run_numbers = [int(pattern.match(x)[1]) for x in os.listdir(output_dir) if pattern.match(x)]
    
    if not run_numbers:
        raise ValueError(f"No run files found in {output_dir}")
    
    run_numbers.sort()

    neighbors_data = []

    for run_num in run_numbers:
        file_path = os.path.join(output_dir, f"run_{run_num}.bp")
        with FileReader(file_path) as reader:
            if "number of neighbors" in reader.available_variables():
                var_info = reader.available_variables()["number of neighbors"]
                steps = int(var_info.get("AvailableStepsCount", 1))
                
                data = reader.read("number of neighbors", step_selection=[steps - 1, 1])
                data = data.flatten()
                neighbors_data.append(data)
            else:
                neighbors_data.append(np.full(7, np.nan))
            
    print(np.array(neighbors_data).shape)
    return np.array(neighbors_data)

def plot_neighbors(neighbors_array, smoothing= True, window_length=10, polyorder=3):
    fig, ax = plt.subplots(figsize=(10, 6))

    for i in range(neighbors_array.shape[1]):
        y = neighbors_array[:, i]
        if smoothing:
            y = savgol_filter(y, window_length, polyorder)
        ax.plot(y, label=f"Shell {i}")
    ax.set_xlabel("Run")
    ax.set_ylabel("Number of Neighbors")
    ax.set_title("Neighbor Analysis (Last Step)")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

output_dir = "/home/hadis/custom_vector/buildParticleOriented/buildVS/jan28_dotTestAdios/outputs/"
neighbors = read_neighbors(output_dir)
plot_neighbors(neighbors)


In [ ]:
def readVariable(file_path, variable_name, start=0, end=100):
    var = FileReader(file_path).read(variable_name, step_selection=[start, end])
    return var

In [ ]:
pos = readVariable(file_path, "positions")

In [ ]:
import numpy as np
from adios2 import Stream
import os

def read_adios_output(outputDir, index, print_summary=True):
    
    data = {}

    with Stream(os.path.join(outputDir, f"run_{index}.bp"), "r") as s:
        for step in s.steps():
            available_vars = s.available_variables()
            for var in available_vars:
                if var not in data:
                    data[var] = []
                data[var].append(s.read(var))

    for var in data:
        data[var] = np.array(data[var])

    if (print_summary):
        print("Data contents summary:\n----------------------")
        for idx, (var, values) in enumerate(data.items(), start=0):
            if values is not None:
                shape = values.shape
                if len(shape) == 1:
                    description = shape[0]
                else:
                    description = f"{shape[0]}*{list(shape[1:])}"
                print(f"{idx}. {var:<40} {description}")

    return list(data.values())

outputDir = "/home/hadis/custom_vector/buildParticleOriented/buildVS/jan28_dotTestAdios/outputs/"
dara = read_adios_output(outputDir, 1)

In [ ]:
simulation_data = read_adios_output(outputDir, 1)


In [ ]:
import matplotlib.pyplot as plt

def plot_snapshot_configuration(step, variable):
    all_steps = variable.shape[0]
    fig, ax = plt.subplots()

    ax.scatter(variable[step, :, 0], variable[step, :, 1], label="initial configuration")
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 20)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"configuration (step {step} of {all_steps})")
    plt.show()
    
def plot_energies(kinetic_energy, potential_energy):
    fig, ax = plt.subplots()

    ax.plot(kinetic_energy[:, 0], label="kinetic energy")
    ax.plot(potential_energy[:]/100, label="potential energy")
    ax.plot(kinetic_energy[:, 0, 0] + kinetic_energy[:, 0, 1] + potential_energy[:]/100, label="total energy")
    ax.plot(kinetic_energy[:, 0, 0] + kinetic_energy[:, 0, 1], label="linear kinetic energy")
    ax.set_xlabel("step")
    ax.set_ylabel("energy")
    ax.set_title("Energies")
    ax.legend()
    plt.show()


In [ ]:
plot_snapshot_configuration(-1, simulation_data[3])

In [ ]:
plot_snapshot_configuration(0, simulation_data[3])

In [ ]:
plot_snapshot_configuration(0, simulation_data[0])

In [ ]:
plot_energies(simulation_data[7], simulation_data[9])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

def plot_neighbors(neighbor_list, smoothing=True, window_length=7, polyorder=2):
    
    fig, ax = plt.subplots(figsize=(7, 4))
    for j in range(1, neighbor_list.shape[2]):
        y = neighbor_list[:,0,j]
        if smoothing:
            y = savgol_filter(y, window_length, polyorder)
        ax.plot(y, label=f'shell {j}')
        
    ax.set_xlabel("Step")
    ax.set_ylabel("Average Number of Neighbors")
    ax.set_title("Neighbor Analysis")
    
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

def plot_temperature (temperature, raw_temperature):
    fig, ax = plt.subplots()

    ax.plot(temperature, label="temperature")
    ax.plot(raw_temperature, label="raw temperature")
    ax.set_xlabel("step")
    ax.set_ylabel("temperature")
    ax.set_title("Temperature")
    ax.legend()
    plt.show()

In [ ]:
plot_neighbors(simulation_data[6])

In [ ]:
def plot_com_velocity(com_linear_velocity):
    fig, ax = plt.subplots()

    ax.plot(com_linear_velocity[:, 0, 0], label="Vx")
    ax.plot(com_linear_velocity[:, 0, 1], label="Vy")
        
    ax.set_xlabel("step")
    ax.set_ylabel("velocity")
    ax.set_title("Center of Mass velocity")
    ax.legend()
    plt.show()
    
def plot_com_angular_velocity(com_angular_velocity):
    fig, ax = plt.subplots()

    ax.plot(com_angular_velocity)
    
    ax.set_xlabel("step")
    ax.set_ylabel("angular velocity")
    ax.set_title("Center of Mass angular velocity")
    plt.show()
    
def plot_average_velocities(velocities):
    fig, ax = plt.subplots()

    ax.plot(np.average(velocities[:, :, 0], axis=1), label="Vx")
    ax.plot(np.average(velocities[:, :, 1], axis=1), label="Vy")
        
    ax.set_xlabel("step")
    ax.set_ylabel("velocity")
    ax.set_title("Center of Mass velocity")
    ax.legend()
    plt.show()   

In [ ]:
plot_average_velocities(simulation_data[2])

In [ ]:
plot_com_velocity(simulation_data[4])
plot_com_angular_velocity(simulation_data[3])